In [ ]:
# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Import Path from pathlib
from pathlib import Path

# Import yfinance package
import yfinance as yf

# Import pandas package
import pandas as pd

# Import pretty printing
from pprint import pprint

In [ ]:
# Load the data issuers data
issuers = pd.read_csv(Path.cwd().parent / 'data' / 'equity_issuers.csv')

In [ ]:
# View the issuers data -> Issuer Name and Security Id
issuers[['Issuer Name', 'Security Id']].head()

In [ ]:
# Pull the data for the first security
stock_data = yf.Ticker("ABB.BO")

In [ ]:
# Extract full of the stock
stock_data_info = stock_data.info

In [ ]:
# Extract only the important information
stock_data_info = {
    'Basic Information': {
        'symbol': stock_data_info.get('symbol'),
        'longName': stock_data_info.get('longName'),
        'currency': stock_data_info.get('currency'),
        'exchange': stock_data_info.get('exchange')
    },

    'Market Data': {
        'currentPrice': stock_data_info.get('currentPrice'),
        'previousClose': stock_data_info.get('previousClose'),
        'open': stock_data_info.get('open'),
        'dayLow': stock_data_info.get('dayLow'),
        'dayHigh': stock_data_info.get('dayHigh'),
        'regularMarketPreviousClose': stock_data_info.get('regularMarketPreviousClose'),
        'regularMarketOpen': stock_data_info.get('regularMarketOpen'),
        'regularMarketDayLow': stock_data_info.get('regularMarketDayLow'),
        'regularMarketDayHigh': stock_data_info.get('regularMarketDayHigh'),
        'fiftyTwoWeekLow': stock_data_info.get('fiftyTwoWeekLow'),
        'fiftyTwoWeekHigh': stock_data_info.get('fiftyTwoWeekHigh'),
        'fiftyDayAverage': stock_data_info.get('fiftyDayAverage'),
        'twoHundredDayAverage': stock_data_info.get('twoHundredDayAverage')
    },

    'Volume and Shares': {
        'volume': stock_data_info.get('volume'),
        'regularMarketVolume': stock_data_info.get('regularMarketVolume'),
        'averageVolume': stock_data_info.get('averageVolume'),
        'averageVolume10days': stock_data_info.get('averageVolume10days'),
        'averageDailyVolume10Day': stock_data_info.get('averageDailyVolume10Day'),
        'sharesOutstanding': stock_data_info.get('sharesOutstanding'),
        'impliedSharesOutstanding': stock_data_info.get('impliedSharesOutstanding'),
        'floatShares': stock_data_info.get('floatShares')
    },

    'Dividends and Yield': {
        'dividendRate': stock_data_info.get('dividendRate'),
        'dividendYield': stock_data_info.get('dividendYield'),
        'payoutRatio': stock_data_info.get('payoutRatio')
    },

    'Valuation and Ratios': {
        'marketCap': stock_data_info.get('marketCap'),
        'enterpriseValue': stock_data_info.get('enterpriseValue'),
        'priceToBook': stock_data_info.get('priceToBook'),
        'debtToEquity': stock_data_info.get('debtToEquity'),
        'grossMargins': stock_data_info.get('grossMargins'),
        'profitMargins': stock_data_info.get('profitMargins')
    },

    'Financial Performance': {
        'totalRevenue': stock_data_info.get('totalRevenue'),
        'revenuePerShare': stock_data_info.get('revenuePerShare'),
        'totalCash': stock_data_info.get('totalCash'),
        'totalCashPerShare': stock_data_info.get('totalCashPerShare'),
        'totalDebt': stock_data_info.get('totalDebt'),
        'earningsGrowth': stock_data_info.get('earningsGrowth'),
        'revenueGrowth': stock_data_info.get('revenueGrowth'),
        'returnOnAssets': stock_data_info.get('returnOnAssets'),
        'returnOnEquity': stock_data_info.get('returnOnEquity')
    },

    'Cash Flow': {
        'freeCashflow': stock_data_info.get('freeCashflow'),
        'operatingCashflow': stock_data_info.get('operatingCashflow')
    },

    'Analyst Targets': {
        'targetHighPrice': stock_data_info.get('targetHighPrice'),
        'targetLowPrice': stock_data_info.get('targetLowPrice'),
        'targetMeanPrice': stock_data_info.get('targetMeanPrice'),
        'targetMedianPrice': stock_data_info.get('targetMedianPrice')
    }
}

In [ ]:
# View the stock info
pprint(stock_data_info)

In [ ]:
# Create dictionary for periods and intervals
periods = {
    '1d': ['1m', '2m', '5m', '15m', '30m', '60m', '90m'],
    '5d': ['1m', '2m', '5m', '15m', '30m', '60m', '90m'],
    '1mo': ['30m', '60m', '90m', '1d'],
    '3mo': ['1d', '5d', '1wk', '1mo'],
    '6mo': ['1d', '5d', '1wk', '1mo'],
    '1y': ['1d', '5d', '1wk', '1mo'],
    '2y': ['1d', '5d', '1wk', '1mo'],
    '5y': ['1d', '5d', '1wk', '1mo'],
    '10y': ['1d', '5d', '1wk', '1mo'],
    'max': ['1d', '5d', '1wk', '1mo'],
}

In [ ]:
# Extract the data for last 1yr with 1d interval
stock_data_hist = stock_data.history(period="2y", interval="1d")
stock_data_hist

In [ ]:
# Clean the data for to keep only the required columns
stock_data_hist = stock_data_hist[['Close']]

In [ ]:
# Imports
import datetime as dt

#Statsmodels is a great library we can use to run regressions
import statsmodels.api as sm

# Import matplotlib for plotting
import matplotlib.pyplot as plt

# Seaborn extends the capabilities of Matplotlib
import seaborn as sns
sns.set_style("darkgrid")

In [ ]:
# Change frequency to day
stock_data_hist = stock_data_hist.asfreq('D', method='ffill')

In [ ]:
# Fill missing values
stock_data_hist = stock_data_hist.fillna(method='ffill')

In [ ]:
# View the new data
stock_data_hist

In [ ]:
# Import the required libraries
from statsmodels.tsa.ar_model import AutoReg

# Define training and testing area
train_df = stock_data_hist.iloc[:int(len(stock_data_hist) * 0.9) + 1] # 90%
test_df = stock_data_hist.iloc[int(len(stock_data_hist) * 0.9):] # 10%

In [ ]:
# Define training model
model = AutoReg(train_df['Close'], 250).fit(cov_type="HC0")

In [ ]:
# Predict data for test data
predictions = model.predict(start=test_df.index[0], end=test_df.index[-1], dynamic=True)

# Predict 90 days into the future
forecast = model.predict(start=test_df.index[0], end=test_df.index[-1]+dt.timedelta(days=90), dynamic=True)

# View the forecasted values
forecast

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from statsmodels.tsa.ar_model import AutoReg
import datetime as dt
train_residuals = train_df['Close'] - ar_model.fittedvalues
test_residuals = test_df['Close'] - ar_predictions

# Prepare data for LSTM
def create_lstm_data(data, time_steps=10):
    X, y = [], []
    for i in range(len(data) - time_steps):
        X.append(data[i:i+time_steps])
        y.append(data[i+time_steps])
    return np.array(X), np.array(y)

# Normalize the residuals
mean_residual = train_residuals.mean()
std_residual = train_residuals.std()
normalized_residuals = (train_residuals - mean_residual) / std_residual

# Create LSTM training data
time_steps = 10
X_train, y_train = create_lstm_data(normalized_residuals.values, time_steps)

# Reshape for LSTM
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))

# Build LSTM model
lstm_model = Sequential([
    LSTM(50, activation='relu', input_shape=(time_steps, 1)),
    Dense(1)
])
lstm_model.compile(optimizer='adam', loss='mse')

# Train LSTM
lstm_model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=1)

# Predict residuals with LSTM
normalized_test_residuals = (test_residuals - mean_residual) / std_residual
X_test, _ = create_lstm_data(normalized_test_residuals.values, time_steps)
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

lstm_predictions = lstm_model.predict(X_test)

# De-normalize LSTM predictions
lstm_predictions = lstm_predictions * std_residual + mean_residual

# Combine AR and LSTM predictions
ar_predictions = ar_predictions[-len(lstm_predictions):]  # Align lengths
final_predictions = ar_predictions + lstm_predictions.flatten()

# Forecast future values
future_steps = 90
ar_forecast = ar_model.predict(
    start=test_df.index[-1] + 1, 
    end=test_df.index[-1] + future_steps, 
    dynamic=True
)

# Prepare future residuals for LSTM forecast
future_residuals = np.zeros(future_steps)  # Assuming no prior residuals for future

# Generate LSTM future predictions
future_lstm_predictions = lstm_model.predict(
    np.zeros((future_steps, time_steps, 1))
).flatten()

In [ ]:
# Combine AR forecast and LSTM future predictions
final_forecast = ar_forecast + (future_lstm_predictions * std_residual + mean_residual)

In [ ]:
# Set style for seaborn plot
sns.set_style('darkgrid')

# Add automatic datetime converters
pd.plotting.register_matplotlib_converters()

# Default figure size
sns.mpl.rc('figure',figsize=(6, 5))

# Set fig and ax
fig, ax = plt.subplots()

# Plot the data
ax = train_df.tail(5).plot(ax=ax, color='blue')
ax = test_df.plot(ax=ax, color='orange')
ax = forecast.plot(ax=ax, color='green')
ax = predictions.plot(ax=ax, color='red')

# Customize the plot
plt.legend(["Train Data", "Test Data", "Forecast", "Predictions"])
plt.title("Stock Price Data")

# Show the plot
plt.show()